In [ ]:
pip install pandas joblib shap numpy matplotlib seaborn scikit-learn

In [ ]:
import os
import joblib
import shap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

In [ ]:
df = pd.read_csv('../data/Diabetes_and_LifeStyle_Dataset_.csv')

In [ ]:
lifestyle_features = [
    "alcohol_consumption_per_week",
    "physical_activity_minutes_per_week",
    "diet_score",
    "sleep_hours_per_day",
    "screen_time_hours_per_day",
    "bmi",
    "waist_to_hip_ratio",
    "systolic_bp",
    "diastolic_bp",
    "heart_rate",
    "cholesterol_total",
    "hdl_cholesterol",
    "ldl_cholesterol",
    "triglycerides",
    "glucose_fasting",
    "glucose_postprandial",
    "insulin_level",
    "hba1c"
    
]
X_seg = df[lifestyle_features]

In [ ]:
scaler = StandardScaler()
X_seg_scaled = scaler.fit_transform(X_seg)
print("Data shape:", X_seg_scaled.shape)

In [ ]:
# K-Means Segmentation

# Apply k-means clustering
kmeans = KMeans(n_clusters=3, random_state=42)
clusters = kmeans.fit_predict(X_seg_scaled)

# Add cluster labels to dataframe
df["lifestyle_segment"] = clusters

# Distribution of patients per segment
print(df["lifestyle_segment"].value_counts())

# Visualize clusters (BMI vs Physical Activity)
sns.scatterplot(
    x=df["bmi"],
    y=df["physical_activity_minutes_per_week"],
    hue=df["lifestyle_segment"],
    palette="Set1"
)
plt.title("Lifestyle Segments (k=3)")
plt.show()

## SHAP Explaination
In order to understand why a patient was assigned to a specific segment, we need to train a Random Forest Surrogate Model.Analyzing this model with SHAP, we are able to rank the health metrics that act as the primary "drivers" for our patient segmentation.

In [ ]:

# Train a lighter RandomForest surrogate model
rf_seg = RandomForestClassifier(
    n_estimators=1000,   # fewer trees for speed
    max_depth=5,       # shallow trees
    random_state=42
)
rf_seg.fit(X_seg_scaled, df["lifestyle_segment"])

# Use TreeExplainer with approximate mode for faster computation
explainer_seg = shap.TreeExplainer(rf_seg)
shap_values_seg = explainer_seg.shap_values(X_seg_scaled, approximate=True)

# Plot top drivers of segmentation (limit to top 10 features)
shap.summary_plot(shap_values_seg, X_seg, plot_type="bar", max_display=10)

In [ ]:
# Save Artifacts
os.makedirs("../artifacts", exist_ok=True)

joblib.dump(rf_seg, "../artifacts/model_2.pkl")
joblib.dump(explainer_seg, "../artifacts/model_2.pkl")

print("SHAP artifacts have been saved successfully.")